In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sklearn as skt
import xgboost as xgb
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
import default_risk.config as cfg
import os
import xgboost as xgb
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.xgboost
import default_risk.config


application_train_df = pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")

load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)

with open(cfg.SCHEMA_JSON, "r") as f:
    schema = json.load(f)

application_train_df.head()


,id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,cnt_children,amt_income_total,amt_credit,amt_annuity,...,flag_document_16,flag_document_18,documents_count,amt_req_credit_berau_hour,amt_req_credit_breau_day,amt_req_credit_breau_week,amt_req_credit_breau_mon,amt_req_credit_breau_qrt,amt_req_credit_breau_year,client_without_querys
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,1,0.0,0.0,0.0,0.0,0.0,1.0,0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,1,0.0,0.0,0.0,0.0,0.0,0.0,0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN,1
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,1,0.0,0.0,0.0,0.0,0.0,0.0,0


In [2]:
Y= application_train_df["target"]
X= application_train_df.drop(columns=["target"])
X.drop(columns=["id_curr"],inplace=True)

categorical_cols = X.select_dtypes(include=['object']).columns
for col in categorical_cols:
    X[col] = X[col].astype('category')

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

hiperparams=  {  
    "objective" : 'binary:logistic',
    "random_state" : 42,
    "eval_metric" :"auc",
    "enable_categorical" : True
}

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"cleaned_dataset_1")

[0]	validation_0-auc:0.71369
[1]	validation_0-auc:0.72214
[2]	validation_0-auc:0.72685
[3]	validation_0-auc:0.72993
[4]	validation_0-auc:0.73181
[5]	validation_0-auc:0.73344
[6]	validation_0-auc:0.73501
[7]	validation_0-auc:0.73769
[8]	validation_0-auc:0.74001
[9]	validation_0-auc:0.74191
[10]	validation_0-auc:0.74342
[11]	validation_0-auc:0.74406
[12]	validation_0-auc:0.74648
[13]	validation_0-auc:0.74719
[14]	validation_0-auc:0.74779
[15]	validation_0-auc:0.74797
[16]	validation_0-auc:0.74837
[17]	validation_0-auc:0.74823
[18]	validation_0-auc:0.74931
[19]	validation_0-auc:0.74960
[20]	validation_0-auc:0.74982
[21]	validation_0-auc:0.75001
[22]	validation_0-auc:0.75034
[23]	validation_0-auc:0.75034
[24]	validation_0-auc:0.75018
[25]	validation_0-auc:0.74978
[26]	validation_0-auc:0.74999
[27]	validation_0-auc:0.74972
[28]	validation_0-auc:0.74942
[29]	validation_0-auc:0.74937
[30]	validation_0-auc:0.74922
[31]	validation_0-auc:0.74987
[32]	validation_0-auc:0.74978
[33]	validation_0-au